In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from matplotlib.transforms import blended_transform_factory
from pathlib import Path

In [ ]:
sample = "dp1_pai24"
save_figs = False

In [ ]:
default_dir = Path(f"../results/samples/{sample}")
fg_cat = pd.read_parquet(default_dir / "foreground.parquet")
bg_cat = pd.read_parquet(default_dir / "background.parquet")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 2), dpi=150)

# Panel 1: FlexZBoost vs LePhare photo-z
settings = dict(extent=(0, 2, 0, 2), gridsize=50, norm="log", edgecolors="none")
ax1.hexbin(
    fg_cat["photoz_fzboost"],
    fg_cat["photoz_lephare"],
    cmap="Greens",
    **settings,
)
ax1.hexbin(
    bg_cat["photoz_fzboost"],
    bg_cat["photoz_lephare"],
    cmap="Purples",
    **settings,
)
ax1.set(
    xlim=(0, 1.5),
    ylim=(0, 1.5),
    xlabel="FlexZBoost photo-z",
    ylabel="LePhare photo-z",
    aspect="equal",
)
ax1.plot([0, 2], [0, 2], c="k", lw=1, ls="--", zorder=0)

# Panel 2: Spec-z vs photo-z
ax2.hexbin(
    fg_cat["spec_z"],
    fg_cat["z_phot"],
    cmap="Greens",
    **settings,
)
ax2.hexbin(
    bg_cat["spec_z"],
    bg_cat["z_phot"],
    cmap="Purples",
    **settings,
)
ax2.set(
    xlim=(0, 1.5),
    ylim=(0, 1.5),
    xlabel="Spec-z",
    ylabel="Photo-z",
    aspect="equal",
)
ax2.plot([0, 2], [0, 2], c="k", lw=1, ls="--", zorder=0)

# Panel 3: Redshift histograms
# Settings for all histograms
settings = dict(range=(0, 1.5), bins=50, density=True, histtype="step")

# Foreground sample
ax3.hist(
    fg_cat["z_phot"],
    **settings,
    color="C2",
    ls="-",
)
ax3.hist(
    fg_cat["spec_z"],
    **settings,
    color="C2",
    ls="--",
)

# Background sample
ax3.hist(
    bg_cat["z_phot"],
    **settings,
    color="C4",
    ls="-",
)
ax3.hist(
    bg_cat["spec_z"],
    **settings,
    color="C4",
    ls="--",
)

ax3.set(
    xlim=(0, 1.5),
    # ylim=(0, 6.5),
    xlabel="Redshift",
    ylabel="Frequency",
    aspect=1.5 / 6.5,
)

ax3.hist([], histtype="step", color="k", ls="-", label="Photo-z")
ax3.hist([], histtype="step", color="k", ls="--", label="Spec-z")
ax3.legend(handlelength=1, frameon=False, fontsize=8, borderpad=0)

fig.subplots_adjust(wspace=0.4)

if save_figs:
    fig.savefig("../figures/fig_photoz_distribution.pdf", dpi=500, bbox_inches="tight")

In [ ]:
fg_red = fg_cat.query("(absmag_g - absmag_r) > 0.5")
fg_blue = fg_cat.query("(absmag_g - absmag_r) < 0.5")

fig, axes = plt.subplots(1, 3, figsize=(7, 2), dpi=200, sharey=True)


# First apparent magnitude
# ------------------------
def njy_to_mag(flux):
    return 31.4 - 2.5 * np.log10(flux)


bins = np.linspace(16, 24, 25)
axes[0].set(xlabel="Apparent $r$-band magnitude", ylabel="Number of galaxies")
axes[0].hist(njy_to_mag(fg_blue["cmodel_flux_r"]), bins=25, alpha=0.5, color="C0")
axes[0].hist(njy_to_mag(fg_red["cmodel_flux_r"]), bins=25, alpha=0.5, color="C3")
axes[0].set(xticks=np.arange(16, 26, 2))  # , ylim=(0, 1e3))

# Vertical bars to indicate comparisons to other surveys
blended = blended_transform_factory(axes[0].transData, axes[0].transAxes)

axes[0].axvline(21, ls="--", c="gray")
axes[0].annotate(
    "",
    xy=(19.2, 0.90),
    xytext=(21, 0.90),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[0].text(
    21.2,
    0.87,
    "Menard,\nMcCleary",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="left",
    va="center",
)

axes[0].axvline(20, ls="--", c="gray")
axes[0].annotate(
    "",
    xy=(18.5, 0.77),
    xytext=(20, 0.77),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[0].text(
    19.8,
    0.80,
    "Genc",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="right",
    va="bottom",
)

axes[0].axvline(17.77, ls="--", c="gray")
axes[0].annotate(
    "",
    xy=(17.77 - 1.5, 0.77),
    xytext=(17.77, 0.77),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[0].text(
    17.57,
    0.80,
    "Peek",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="right",
    va="bottom",
)

# Now stellar mass
# ------------------------
bins = np.geomspace(0.6e8, 4e11, 25)
axes[1].set(xlabel="Stellar mass [$M_\\odot$]", xscale="log")
axes[1].hist(
    10 ** fg_blue["stellar_mass_log"],
    bins=bins,
    alpha=0.5,
    color="C0",
)
axes[1].hist(
    10 ** fg_red["stellar_mass_log"],
    bins=bins,
    alpha=0.5,
    color="C3",
)

# Vertical bars to indicate comparisons to other surveys
blended = blended_transform_factory(axes[1].transData, axes[1].transAxes)

axes[1].axvline(10**9, ls="--", c="gray")
axes[1].annotate(
    "",
    xy=(10**10, 0.77),
    xytext=(10**9, 0.77),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[1].text(
    10**9.1,
    0.80,
    "Peek,\nMcCleary",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="left",
    va="bottom",
)

axes[1].axvline(10**10.2, ls="--", c="gray")
axes[1].annotate(
    "",
    xy=(10**11.2, 0.77),
    xytext=(10**10.2, 0.77),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[1].text(
    10**10.3,
    0.80,
    "Menard,\nGenc",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="left",
    va="bottom",
)

# Finally halo mass
# ------------------------
axes[2].set(xlabel="Halo mass [$M_\\odot$]", xscale="log")
bins = np.geomspace(3e10, 1e14, 25)
axes[2].hist(
    10 ** fg_blue["halo_mass_log"],
    bins=bins,
    alpha=0.5,
    color="C0",
)
axes[2].hist(
    10 ** fg_red["halo_mass_log"],
    bins=bins,
    alpha=0.5,
    color="C3",
)

# Vertical bars to indicate characteristic halo mass of samples
blended = blended_transform_factory(axes[2].transData, axes[2].transAxes)
"""axes[2].axvline(4.2e12, ls="--", c="C0")
axes[2].text(
    0.9 * 4.2e12,
    0.72,
    "McCleary\nblue",
    transform=blended,
    color="C0",
    fontsize=7,
    ha="right",
    va="bottom",
)"""
axes[2].axvline(10**13.7, ls="--", c="C3")
axes[2].text(
    0.85 * 10**13.7,
    0.80,
    "McCleary\nredMaGiC",
    transform=blended,
    color="C3",
    fontsize=7,
    ha="right",
    va="bottom",
)

if save_figs:
    fig.savefig("../figures/fig_foreground_properties.pdf", bbox_inches="tight")

In [ ]:
joint = pd.concat([fg_blue, fg_red])
joint["set"] = ["blue"] * len(fg_blue) + ["red"] * len(fg_red)

jp = sns.jointplot(
    x=joint["absmag_r"],
    y=joint["absmag_g"] - joint["absmag_r"],
    hue=joint["set"],
    palette=["C0", "C3"],
    kind="kde",
    legend=False,
    levels=8,
    joint_kws={"thresh": 0.1},
    xlim=(-22.6, -16.75),
    ylim=(0.1, 0.9),
)

fig = jp.figure
fig.set_figwidth(2.5)
fig.set_figheight(2)
fig.set_dpi(200)

# Set axis labels
jp.set_axis_labels("Absolute $r$-band magnitude", "Rest-frame $g - r$ color")

if save_figs:
    jp.savefig("../figures/fig_foreground_cmd.pdf", bbox_inches="tight")

In [ ]:
# Median halo mass of the red sample
10 ** fg_red["halo_mass_log"].median() / 1e11

In [ ]:
10 ** fg_cat["halo_mass_log"].median() / 1e11

In [ ]:
10 ** fg_cat["stellar_mass_log"].median() / 1e9

In [ ]:
10 ** fg_red["halo_mass_log"].median() / 1e11